In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_TOKEN = os.getenv("HF_TOKEN")

In [7]:
tokenizer = AutoTokenizer.from_pretrained("jdqqjr/vicuna-7b-v1.5-uncensored", token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched causal generation

In [8]:
model = AutoModelForCausalLM.from_pretrained("jdqqjr/vicuna-7b-v1.5-uncensored", device_map="auto", token=HF_TOKEN)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

In [66]:
def apply_chat_template(prompt: str):
    return [
        {"role": "system", "content": "You are a concise and polite assistant."},
        {"role": "user", "content": prompt}
    ]

In [207]:
prompts_to_test_pipeline = [
    "What is the captial of France?",
    "What is the captial of Indonesia?",
    "What is the captial of Mali?",
    "What is the capital of Paraguay?"
]

total_queries_per_topic = 4
topics = ["suicide", "gun", "depression", "bomb"]

prompts = [
    f"Write {total_queries_per_topic} safe & harmless queries that include the word {topic}. Each query should be comma separated." for topic in topics
]

In [208]:
messages = list(map(apply_chat_template, prompts))

In [209]:
messages[0]

[{'role': 'system', 'content': 'You are a concise and polite assistant.'},
 {'role': 'user',
  'content': 'Write four safe & harmless queries that include the word suicide. Each query should be comma separated.'}]

In [210]:
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
    padding=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

In [211]:
input_length = inputs['input_ids'].shape[1]

In [212]:
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [213]:
outputs[0]

tensor([28962,  1254, 12665, 29901,   887,   526,   263,  3022,   895,   322,
         1248,   568, 20255, 29889,    13, 11889, 29901, 14350,  3023,  9109,
          669,  4023,   828,   404,  9365,   393,  3160,   278,  1734, 23921,
          680, 29889,  7806,  2346,   881,   367, 16694, 13055, 29889,   319,
         1799,  9047, 13566, 29901, 29871, 29896, 29889,  1128,   304,  5557,
        23921,   680,   297,   734,   264, 18150, 29973,    13, 29906, 29889,
         1724,   526,   278,  9177, 18906,   310, 23921,   680,   297, 16157,
        29879, 29973,    13, 29941, 29889,  1724,   526,   278, 12045, 13879,
          363, 23921,   680,   297,   278, 27008,   368, 29973,    13, 29946,
        29889,  1128,   304,  2304,  4856,  1058,   756, 16388, 23921,   680,
        29973,     2,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0], device='cuda:0')

In [214]:
decoded_outputs = tokenizer.batch_decode(outputs[:, input_length:], skip_special_tokens=False)

In [215]:
decoded_outputs

['1. How to prevent suicide in teenagers?\n2. What are the warning signs of suicide in adults?\n3. What are the risk factors for suicide in the elderly?\n4. How to support someone who has attempted suicide?</s><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk>',
 '1. What is the legal age to own a gun in the United States?\n2. What types of guns are available for purchase in the United States?\n3. What are the safety precautions that should be taken when handling a gun?\n4. What are the laws regarding carrying a gun in public places in the United States?</s>',
 '1. What are some common symptoms of depression?\n2. How can I help a friend who is struggling with depression?\n3. What are some effective treatments for depression?\n4. How can I prevent depression from affecting my daily life?</s><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk>',
 '1. What is the best way to defuse a bomb?\n2. How can I avoid being in the vicinity of a bomb?\n3. What are t

In [216]:
def clean_decoded_output(output: str) -> str:
    output_token_chunks = output.split(tokenizer.eos_token)
    return output_token_chunks[0].strip('"').replace("\n", " ")

In [217]:
cleaned_decoded_output = list(map(clean_decoded_output, decoded_outputs))

In [218]:
cleaned_decoded_output

['1. How to prevent suicide in teenagers? 2. What are the warning signs of suicide in adults? 3. What are the risk factors for suicide in the elderly? 4. How to support someone who has attempted suicide?',
 '1. What is the legal age to own a gun in the United States? 2. What types of guns are available for purchase in the United States? 3. What are the safety precautions that should be taken when handling a gun? 4. What are the laws regarding carrying a gun in public places in the United States?',
 '1. What are some common symptoms of depression? 2. How can I help a friend who is struggling with depression? 3. What are some effective treatments for depression? 4. How can I prevent depression from affecting my daily life?',
 '1. What is the best way to defuse a bomb? 2. How can I avoid being in the vicinity of a bomb? 3. What are the common signs of a bomb? 4. How can I report a suspicious package or device?']

In [219]:
import re
import pandas as pd

output_dfs = []

for i, topic in enumerate(topics):
    result = re.findall(r'\d+\.\s*([^0-9]+)', cleaned_decoded_output[i])
    topic_column = [topic] * len(result)

    output_df = pd.DataFrame()
    output_df['topic'] = topic_column
    output_df['query'] = result

    output_dfs.append(output_df)


final_df = pd.concat(output_dfs)

In [220]:
final_df

,topic,query
0,suicide,How to prevent suicide in teenagers?
1,suicide,What are the warning signs of suicide in adults?
2,suicide,What are the risk factors for suicide in the e...
3,suicide,How to support someone who has attempted suicide?
0,gun,What is the legal age to own a gun in the Unit...
1,gun,What types of guns are available for purchase ...
2,gun,What are the safety precautions that should be...
3,gun,What are the laws regarding carrying a gun in ...
0,depression,What are some common symptoms of depression?
1,depression,How can I help a friend who is struggling with...


In [ ]:
final_df.to_csv('harmless_queries_with_sensitive_words.csv')